# Example 4 — `persist_db=True` with In-Memory and Disk-Persist Data

`persist_db=True` keeps the builder's scratch DuckDB file alive across `extract_segments` → `evaluate_final_coverage` instead of deleting it after each call. This avoids re-materialising the dataset into the scratch on every method call.

This notebook demonstrates both data-loading paths:

1. **In-memory PyArrow** — data fits in RAM, use `UniversalDataLoader.load()`.
2. **On-disk DuckDB** — data too large for RAM, use `stream_to_duckdb()` with an explicit `db_path`.

Variant A uses the **chained `with StrategicSegmentBuilder(...) as b:`** idiom from the README — one block that extracts, evaluates, health-checks, and scores against the same scratch artifact, with `b.close()` running automatically on exit.

## 1. SETUP

In [1]:
import rapidsegment as rs
import pandas as pd
import duckdb
import os
from rapidsegment import UniversalDataLoader, StrategicSegmentBuilder, StrategicSegmentScore
pd.options.display.max_columns = 100

In [2]:
print(f"RapidSegment version: {rs.__version__}")

RapidSegment version: 1.3.2


---

## 2. Variant A — In-Memory PyArrow (small data)

Load the CSV into a PyArrow table in memory and binarise the target.

In [3]:
data_path = None
for c in [
    "/teamspace/studios/this_studio/RapidSegment/Notebooks/Term_Deposit_Sub/bank_train.csv",
    os.path.join("..", "Term_Deposit_Sub", "bank_train.csv"),
    "Term_Deposit_Sub/bank_train.csv",
]:
    if os.path.exists(c):
        data_path = c
        break
if not data_path:
    raise FileNotFoundError("bank_train.csv not found — add it to Notebooks/Term_Deposit_Sub/")

target_col = "y"

# Load as in-memory PyArrow table (no disk persist)
data = UniversalDataLoader(file_path=data_path).load()
print(f"Loaded {data.num_rows} rows, {data.num_columns} columns")

# Binarise via DuckDB SQL (lightweight temp connection)
con = duckdb.connect(":memory:")
con.register("train_raw", data)
data = con.execute(f"""
SELECT *,
    CAST(CASE WHEN "{target_col}" = 'yes' THEN 1.0 ELSE 0.0 END AS DOUBLE) AS {target_col}_binary,
    CAST(CASE WHEN "default"         = 'yes' THEN 1.0 ELSE 0.0 END AS DOUBLE) AS default_binary,
    CAST(CASE WHEN "housing"         = 'yes' THEN 1.0 ELSE 0.0 END AS DOUBLE) AS housing_binary,
    CAST(CASE WHEN "loan"            = 'yes' THEN 1.0 ELSE 0.0 END AS DOUBLE) AS loan_binary
FROM train_raw
""").fetch_arrow_table()
con.close()
print("Binarised columns added:", [c for c in data.column_names if "binary" in c])

2026-09-13 15:01:22,390 | INFO     | [data_loader.py:151] | 📂 Loading file: /teamspace/studios/this_studio/RapidSegment/Notebooks/Term_Deposit_Sub/bank_train.csv (extension: .csv)


Loaded 40689 rows, 17 columns
Binarised columns added: ['y_binary', 'default_binary', 'housing_binary', 'loan_binary']


/tmp/ipykernel_5663/434913295.py:29: DeprecationWarning: fetch_arrow_table() is deprecated, use to_arrow_table() instead.
  """).fetch_arrow_table()


## 3. Chained pipeline — everything inside one `with` block

The scratch database (auto-created under `experiments/`) is **kept alive** for the duration of the `with` block. Extract → evaluate → health → score all share the same file, then `b.close()` runs automatically when the block exits.

In [4]:
with StrategicSegmentBuilder(
    target=target_col + "_binary",
    min_sample_size=100,
    min_lift=1.0,
    min_events=50,
    top_n_vars=10,
    max_segments=10,
    param_grid={"min_sample_size": [20000, 15000, 10000, 5000, 500], "min_lift": [3.0, 2.0, 1.5]},
    enable_diversity=False,
    max_feature_reuse=5,
    enable_1way=True, enable_2way=True, enable_3way=True,
    ignore_features=[target_col, "loan", "default", "housing"],
    sort_priority="lift_rate_count",
    binning_method="optimal_cart",
    selection_metric="iv",
    db_path=None,       # auto scratch (separate from data)
    persist_db=True,    # keep scratch alive for the whole block
) as b:
    print("b.db_path BEFORE extract:", b.db_path)          # None -> auto-created on first call

    segments_df = b.extract_segments(data)
    final_eval = b.evaluate_final_coverage(data)           # same DB file, reuses original_df
    health = b.generate_feature_health_report(data, ["age", "balance", "duration"])

    print("b.db_path AFTER  extract :", b.db_path)          # experiments/segmentation_*.duckdb

    # Build PREDICTED (original columns + seg_N flags + row ID) inside the scratch
    df_seg = pd.DataFrame(segments_df)
    seg_cases = "\n".join(
        f'CASE WHEN ({row["sql_filter"]}) THEN 1 ELSE 0 END AS seg_{i+1},'
        for i, row in enumerate(df_seg.to_dict("records"))
    )
    con = duckdb.connect(b.db_path)
    con.execute(f"""
CREATE OR REPLACE TABLE PREDICTED AS (
SELECT *,
    {seg_cases}
    ROW_NUMBER() OVER () AS ID
FROM original_df
)
""")
    predicted = con.execute("SELECT * FROM PREDICTED").fetch_arrow_table()
    con.close()

    # Scorecard from the persisted scratch
    seg_cols = [f"seg_{i+1}" for i in range(len(df_seg))]
    scorer = StrategicSegmentScore(target_col + "_binary", "ID", seg_cols)
    model = scorer.calculate_and_export_weights(
        predicted,                   # in-memory PyArrow (no file path needed)
        "model_a.json",
        db_path=b.db_path,           # scorer uses the same scratch for its compute
    )

print("After with-block, b.db_path:", b.db_path)          # None -> auto close() ran
print("PREDICTED rows:", predicted.num_rows, "| columns:", predicted.num_columns)

2026-09-13 15:01:22,535 | INFO     | [builder.py:1228] | 🚀 Starting hierarchical segment extraction...
2026-09-13 15:01:22,540 | INFO     | [builder.py:1253] | 📂 Created temporary disk-backed DB at: experiments/segmentation_20260913_48c25a12.duckdb
2026-09-13 15:01:22,572 | INFO     | [builder.py:1277] | ⚙️ DuckDB Configured for Disk Spilling: Threads=4/4, MemoryLimit=11GB, TempDir=experiments/tmp_20260913_48c25a12
2026-09-13 15:01:22,573 | INFO     | [builder.py:1281] | 📊 Sort priority: lift_rate_count
2026-09-13 15:01:22,574 | INFO     | [builder.py:1282] | 📦 Binning method: optimal_cart


b.db_path BEFORE extract: None


2026-09-13 15:01:22,765 | INFO     | [builder.py:1365] | 📊 Dynamic Grid Search Enabled: 15 configurations.
2026-09-13 15:01:22,768 | INFO     | [builder.py:1373] | 🔒 Locking Original Base Rate: 11.70%
2026-09-13 15:01:22,771 | INFO     | [builder.py:1399] | 🔄 Iteration 1 | Remaining Volume: 40,689 | Base Rate: 11.70%
2026-09-13 15:01:22,771 | INFO     | [builder.py:340] | 🔍 Computing IV and bins for 16 features...
2026-09-13 15:01:25,202 | INFO     | [builder.py:1635] | 📌 Feature Usage Tracker Update -> 'housing_binary' used count = 1
2026-09-13 15:01:25,204 | INFO     | [builder.py:1635] | 📌 Feature Usage Tracker Update -> 'month' used count = 1
2026-09-13 15:01:25,204 | INFO     | [builder.py:1635] | 📌 Feature Usage Tracker Update -> 'poutcome' used count = 1
2026-09-13 15:01:25,205 | INFO     | [builder.py:1657] | ✅ Segment 1 Captured (Size Floor: 500 | Lift Floor: 3.0): rows=506, events=317.0, lift=5.36
  Rule: housing_binary=(-inf, 0.50) & month=[apr, oct, dec, sep, mar] & poutcom

b.db_path AFTER  extract : experiments/segmentation_20260913_48c25a12.duckdb


/tmp/ipykernel_5663/4021333756.py:42: DeprecationWarning: fetch_arrow_table() is deprecated, use to_arrow_table() instead.
  predicted = con.execute("SELECT * FROM PREDICTED").fetch_arrow_table()
2026-09-13 15:01:47,546 | INFO     | [scorer.py:86] | 🚀 Initialising out‑of‑core DuckDB scorecard engine...
2026-09-13 15:01:47,845 | INFO     | [scorer.py:153] | 📊 Computing scorecard weights...
2026-09-13 15:01:47,847 | INFO     | [scorer.py:206] | ✅ Scorecard has 10 distinct score values – good for deciling.
2026-09-13 15:01:47,848 | INFO     | [scorer.py:211] | ⚡ Scoring population natively via SQL engine...
2026-09-13 15:01:47,928 | INFO     | [scorer.py:231] | 📉 Dataset Zero‑Inflation Rate: 88.30%
2026-09-13 15:01:47,930 | INFO     | [scorer.py:236] | 📈 Calibrating deciles across active populations...
2026-09-13 15:01:47,942 | INFO     | [scorer.py:284] | ✅ Scorecard exported to: model_a.json


After with-block, b.db_path: None
PREDICTED rows: 40689 | columns: 32


In [5]:
print(pd.DataFrame(final_eval).to_string(index=False))

 segment  total_count  target_events  response_rate  base_response_rate  capture_rate     lift  cumulative_sample_capture  cumulative_event_capture
       1          506          317.0      62.648221           11.698493      1.243579 5.355238                   1.243579                  6.659664
       2          739          442.0      59.810555           11.698493      1.816216 5.112672                   3.059795                 15.945378
       3          650          368.0      56.615385           11.698493      1.597483 4.839545                   4.657278                 23.676471
       4          861          460.0      53.426249           11.698493      2.116051 4.566934                   6.773329                 33.340336
       5          542          251.0      46.309963           11.698493      1.332055 3.958626                   8.105385                 38.613445
       6         1257          519.0      41.288783           11.698493      3.089287 3.529410                  

In [6]:
print()
print("Segment weights (from model_a.json):")
for k, v in model["segment_weights"].items():
    print(f"  {k}: weight={v['weight']}")


Segment weights (from model_a.json):
  seg_1: weight=63
  seg_2: weight=60
  seg_3: weight=61
  seg_4: weight=55
  seg_5: weight=54
  seg_6: weight=48
  seg_7: weight=44
  seg_8: weight=46
  seg_9: weight=43
  seg_10: weight=36


---

## 4. Variant B — On-Disk DuckDB (large data)

The same pipeline, but the data lives in a named persistent `.duckdb` file via `stream_to_duckdb(db_path=...)`. Here `b.close()` is called **manually** — the equivalent of what the `with` block does automatically in Variant A.

In [7]:
PERSIST_DIR = os.path.join("data", "example4")
os.makedirs(PERSIST_DIR, exist_ok=True)
db_file = os.path.join(PERSIST_DIR, "bank_data.duckdb")
if os.path.exists(db_file):
    os.remove(db_file)

out = UniversalDataLoader(file_path=data_path).stream_to_duckdb(
    db_path=db_file, table_name="bank_train"
)
print("Persistent DB:", out)

Persistent DB: /teamspace/studios/this_studio/RapidSegment/Notebooks/Examples/data/example4/bank_data.duckdb


In [8]:
con = duckdb.connect(out)
con.execute(f"""
CREATE OR REPLACE TABLE bank_train AS (
SELECT *,
    CAST(CASE WHEN "{target_col}" = 'yes' THEN 1.0 ELSE 0.0 END AS DOUBLE) AS {target_col}_binary,
    CAST(CASE WHEN "default"      = 'yes' THEN 1.0 ELSE 0.0 END AS DOUBLE) AS default_binary,
    CAST(CASE WHEN "housing"      = 'yes' THEN 1.0 ELSE 0.0 END AS DOUBLE) AS housing_binary,
    CAST(CASE WHEN "loan"         = 'yes' THEN 1.0 ELSE 0.0 END AS DOUBLE) AS loan_binary
FROM bank_train
)
""")
print(con.sql("SELECT * FROM bank_train LIMIT 2"))
con.close()

┌────────┬─────────────┬─────────┬───────────┬─────────┬─────────┬─────────┬─────────┬──────────┬────────┬─────────┬──────────┬──────────┬────────┬──────────┬──────────┬─────────┬──────────┬────────────────┬────────────────┬─────────────┐
│  age   │     job     │ marital │ education │ default │ balance │ housing │  loan   │ contact  │  day   │  month  │ duration │ campaign │ pdays  │ previous │ poutcome │    y    │ y_binary │ default_binary │ housing_binary │ loan_binary │
│ double │   varchar   │ varchar │  varchar  │ boolean │ double  │ boolean │ boolean │ varchar  │ double │ varchar │  double  │  double  │ double │  double  │ varchar  │ boolean │  double  │     double     │     double     │   double    │
├────────┼─────────────┼─────────┼───────────┼─────────┼─────────┼─────────┼─────────┼──────────┼────────┼─────────┼──────────┼──────────┼────────┼──────────┼──────────┼─────────┼──────────┼────────────────┼────────────────┼─────────────┤
│   46.0 │ blue-collar │ married │ primary  

## 5. Extract + Evaluate + Score (manual `close()`)

`extract_segments(out)` attaches the persistent DB read-only as `__rs_src`; the builder's scratch is a **separate** file with `persist_db=True`. Note the manual `b.close()` after scoring — the `with`-block in Variant A does this for you.

In [9]:
b = StrategicSegmentBuilder(
    target=target_col + "_binary",
    min_sample_size=100,
    min_lift=1.0,
    min_events=50,
    top_n_vars=10,
    max_segments=10,
    param_grid={"min_sample_size": [20000, 15000, 10000, 500, 500], "min_lift": [3.0, 2.0, 1.5]},
    enable_diversity=False,
    max_feature_reuse=5,
    enable_1way=True, enable_2way=True, enable_3way=True,
    ignore_features=[target_col, "loan", "default", "housing"],
    sort_priority="lift_rate_count",
    binning_method="optimal_cart",
    selection_metric="iv",
    db_path=None,
    persist_db=True,
)

segments_df_b = b.extract_segments(out)
final_eval_b = b.evaluate_final_coverage(out)
print(pd.DataFrame(final_eval_b).to_string(index=False))

2026-09-13 15:01:49,975 | INFO     | [builder.py:1228] | 🚀 Starting hierarchical segment extraction...
2026-09-13 15:01:49,978 | INFO     | [builder.py:1253] | 📂 Created temporary disk-backed DB at: experiments/segmentation_20260913_bd4c7795.duckdb
2026-09-13 15:01:50,053 | INFO     | [builder.py:1277] | ⚙️ DuckDB Configured for Disk Spilling: Threads=4/4, MemoryLimit=11GB, TempDir=experiments/tmp_20260913_bd4c7795
2026-09-13 15:01:50,054 | INFO     | [builder.py:1281] | 📊 Sort priority: lift_rate_count
2026-09-13 15:01:50,055 | INFO     | [builder.py:1282] | 📦 Binning method: optimal_cart
2026-09-13 15:01:50,334 | INFO     | [builder.py:1365] | 📊 Dynamic Grid Search Enabled: 15 configurations.
2026-09-13 15:01:50,339 | INFO     | [builder.py:1373] | 🔒 Locking Original Base Rate: 11.70%
2026-09-13 15:01:50,343 | INFO     | [builder.py:1399] | 🔄 Iteration 1 | Remaining Volume: 40,689 | Base Rate: 11.70%
2026-09-13 15:01:50,345 | INFO     | [builder.py:340] | 🔍 Computing IV and bins for 

 segment  total_count  target_events  response_rate  base_response_rate  capture_rate     lift  cumulative_sample_capture  cumulative_event_capture
       1          506          317.0      62.648221           11.698493      1.243579 5.355238                   1.243579                  6.659664
       2          739          442.0      59.810555           11.698493      1.816216 5.112672                   3.059795                 15.945378
       3          650          368.0      56.615385           11.698493      1.597483 4.839545                   4.657278                 23.676471
       4          861          460.0      53.426249           11.698493      2.116051 4.566934                   6.773329                 33.340336
       5          542          251.0      46.309963           11.698493      1.332055 3.958626                   8.105385                 38.613445
       6         1257          519.0      41.288783           11.698493      3.089287 3.529410                  

In [10]:
df_seg_b = pd.DataFrame(segments_df_b)
seg_cases_b = "\n".join(
    f'CASE WHEN ({row["sql_filter"]}) THEN 1 ELSE 0 END AS seg_{i+1},'
    for i, row in enumerate(df_seg_b.to_dict("records"))
)

predicted_data = "PREDICTED"
con = duckdb.connect(out)
con.execute(f"""
CREATE OR REPLACE TABLE {predicted_data} AS (
SELECT *,
    {seg_cases_b}
    ROW_NUMBER() OVER () AS ID
FROM bank_train
)
""")
con.close()

seg_cols_b = [f"seg_{i+1}" for i in range(len(df_seg_b))]
scorer = StrategicSegmentScore(target_col + "_binary", "ID", seg_cols_b)
model_b = scorer.calculate_and_export_weights(
    out,                                  # data file (read-only attach)
    "model_b.json",
    db_path=b.db_path,                    # scorer reuses the builder's scratch
    table_name=predicted_data,
)

print()
print("Segment weights (from model_b.json):")
for k, v in model_b["segment_weights"].items():
    print(f"  {k}: weight={v['weight']}")

2026-09-13 15:02:13,313 | INFO     | [scorer.py:86] | 🚀 Initialising out‑of‑core DuckDB scorecard engine...


2026-09-13 15:02:13,577 | INFO     | [scorer.py:153] | 📊 Computing scorecard weights...
2026-09-13 15:02:13,578 | INFO     | [scorer.py:206] | ✅ Scorecard has 10 distinct score values – good for deciling.
2026-09-13 15:02:13,579 | INFO     | [scorer.py:211] | ⚡ Scoring population natively via SQL engine...
2026-09-13 15:02:13,728 | INFO     | [scorer.py:231] | 📉 Dataset Zero‑Inflation Rate: 88.30%
2026-09-13 15:02:13,731 | INFO     | [scorer.py:236] | 📈 Calibrating deciles across active populations...
2026-09-13 15:02:13,738 | INFO     | [scorer.py:284] | ✅ Scorecard exported to: model_b.json



Segment weights (from model_b.json):
  seg_1: weight=63
  seg_2: weight=60
  seg_3: weight=61
  seg_4: weight=55
  seg_5: weight=54
  seg_6: weight=48
  seg_7: weight=44
  seg_8: weight=46
  seg_9: weight=43
  seg_10: weight=36


In [11]:
b.close()                                  # manual cleanup (what `with` does for you)
print("Scratch released:", b.db_path)
print("Persistent data file:", os.path.abspath(out))

Scratch released: None
Persistent data file: /teamspace/studios/this_studio/RapidSegment/Notebooks/Examples/data/example4/bank_data.duckdb


In [12]:
# Optional: remove the persistent data file (like Example 3 does)
if os.path.exists(out):
    os.remove(out)
    print("Removed:", out)

Removed: /teamspace/studios/this_studio/RapidSegment/Notebooks/Examples/data/example4/bank_data.duckdb


## Summary

| Variant | Lifecycle | Data Source | Builder Input | Scorer Input |
|---------|-----------|-------------|---------------|--------------|
| **A (chained)** | `with b:` → auto `close()` | `.load()` → PyArrow table | `b.extract_segments(arrow_table)` | `calculate_and_export_weights(arrow_table, ..., db_path=b.db_path)` |
| **B (manual)** | `b.close()` by hand | `.stream_to_duckdb(db_path=...)` → `.duckdb` path | `b.extract_segments(db_path)` | `calculate_and_export_weights(db_path, ..., table_name=..., db_path=b.db_path)` |

**Key rules:**

- `builder.db_path` must **never** equal the data file — same-process config conflict (DuckDB rejects it).
- `persist_db=True` keeps the scratch alive; use `with StrategicSegmentBuilder(...) as b:` (Variant A) for automatic cleanup, or call `b.close()` manually (Variant B).
- The scorer's `db_path` can point to the builder's scratch — it creates its own tables inside it without conflict.